# Interview AI - Qwen 2.5 1.5B Fine-Tuning (Robust)

Use this notebook to train your model on Google Colab's Free GPU (Tesla T4).

### Steps:
1. **Runtime** -> **Disconnect and Delete Runtime**.
2. **Runtime** -> **Change runtime type** -> Select **T4 GPU**.
3. Run the cells below in order.
4. When asked, upload your `fine_tune_samples.jsonl` file.
5. Wait for training (~15-20 mins).
6. Download the `finetuned_llm.zip` file.

In [ ]:
# [1] Install Dependencies
!pip install -q torch transformers peft bitsandbytes datasets accelerate

In [ ]:
# [2] Upload Data
from google.colab import files
import os

# DISABLE W&B to prevent hangs/crashes
os.environ["WANDB_DISABLED"] = "true"

# Check if file already exists (from previous upload)
filename = "fine_tune_samples.jsonl"

if not os.path.exists(filename):
    print("Please upload 'fine_tune_samples.jsonl'...")
    uploaded = files.upload()
    if uploaded:
        filename = next(iter(uploaded))
    print(f"Uploaded: {filename}")
else:
    print(f"Found existing file: {filename}")

In [ ]:
# [3] Training Script
import json
import torch
import gc
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Safety Check for filename
if 'filename' not in globals():
    filename = "fine_tune_samples.jsonl"
    if not os.path.exists(filename):
         raise FileNotFoundError("Please run Cell [2] and upload the file first!")

# Configuration
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "finetuned_llm"

# Clean Memory
torch.cuda.empty_cache()
gc.collect()

# Custom Callback
class PrinterCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"[Epoch {state.epoch:.2f}] Step {state.global_step}/{state.max_steps} | Loss: {logs['loss']:.4f}")

def load_training_data(data_path):
    texts = []
    with open(data_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)
            if "messages" in item:
                user = item["messages"][0]["content"]
                asst = item["messages"][1]["content"]
                text = f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n{asst}<|im_end|>"
            else:
                instr = item.get("instruction", "")
                Inp = item.get("input", "")
                out = item.get("output", "")
                if isinstance(Inp, dict):
                    Inp = "\n".join([f"{k.title()}: {v}" for k, v in Inp.items()])
                if isinstance(out, dict):
                    out = json.dumps(out, indent=2)
                text = f"<|im_start|>user\n{instr}\n\n{Inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
            texts.append({"text": text})
    return Dataset.from_list(texts)

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# CRITICAL: Enable Gradients
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Load Data & Tokenize (Max Length 512)
dataset = load_training_data(filename)
dataset = dataset.map(lambda x: tokenizer(x["text"], truncation=True, max_length=512, padding="max_length"), batched=True)

print("Starting training (Robust)...")
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to=["none"],
    gradient_checkpointing=True
)

from transformers import Trainer
trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[PrinterCallback]
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("Training Complete!")

In [ ]:
# [4] Zip and Download
!zip -r finetuned_llm.zip finetuned_llm
files.download('finetuned_llm.zip')